<a href="https://colab.research.google.com/github/JAllen48-eng/ASHP_Model/blob/main/Air_source_Heat_Pump.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
=============================================================================
Project: A Computational Framework for Assessing Grid Impact of ASHPs
Module: ENGG30051 Individual Engineering Project
Author: James Allen (N1148244)
Date: April 2026

Description:
This script contains the object-oriented simulation framework (1R1C network)
and proportional inverter control logic used to model residential heat pump
demand under varying thermal resistances.
=============================================================================
"""

In [ ]:
import numpy as np

class ASHP:
    def __init__(self, hp_power, R, C, dt):
        """
        Initializes the Air Source Heat Pump and 1R1C Building Thermal Model.
        """
        self.hp_power = hp_power
        self.R = R
        self.C = C
        self.dt = dt

    def calculate_cop(self, T_out, flow_temp):
        """
        Calculates COP based on Carnot efficiency and flow temperature.
        """
        T_hot_K = flow_temp + 273.15
        T_cold_K = T_out + 273.15

        if T_hot_K <= T_cold_K:
            theoretical_cop = 5.0
        else:
            theoretical_cop = T_hot_K / (T_hot_K - T_cold_K)

        # 0.4 is the practical efficiency factor of the compressor
        return min(5.0, max(1.0, 0.4 * theoretical_cop))

    def run_ashp_simulation_advanced(self, T_out_data, T_min=19.0, T_max=21.0, flow_temp=45.0):
        """
        Advanced simulation featuring an inverter-driven compressor,
        flow-temperature COP physics, and a thermostat deadband (interval).
        """
        num_steps = len(T_out_data)
        T_in = np.zeros(num_steps + 1)
        P_in = np.zeros(num_steps)
        COP_array = np.zeros(num_steps)

        # Start the simulation right in the middle of our comfort zone
        T_in[0] = (T_min + T_max) / 2
        k_p = 1000  # Proportional gain for ramping up

        for t in range(num_steps):
            current_COP = self.calculate_cop(T_out_data[t], flow_temp)
            COP_array[t] = current_COP

            # --- THERMOSTAT DEADBAND LOGIC ---
            if T_in[t] < T_min:
                # Too cold! Ramp up power proportional to the error
                P_in[t] = min(self.hp_power, (T_min - T_in[t]) * k_p)

            elif T_in[t] > T_max:
                # Too hot! Turn the compressor off completely.
                P_in[t] = 0

            else:
                # Inside the comfort band.
                # Calculate power needed to match the building heat loss.
                heat_loss_watts = (T_in[t] - T_out_data[t]) / self.R
                required_power = heat_loss_watts / current_COP
                P_in[t] = max(0, min(self.hp_power, required_power))

            # 1R1C Thermal Physics equations
            inertia = (1 - self.dt / (self.R * self.C)) * T_in[t]
            heating = (self.dt * current_COP / self.C) * P_in[t]
            disturbance = (self.dt / (self.R * self.C)) * T_out_data[t]

            T_in[t+1] = inertia + heating + disturbance

        return T_in, P_in, COP_array

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

# Ensure weather data is loaded into T_out_array
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/ninja_weather_52.9108_-1.1860_uncorrected.csv', skiprows=3)
T_out_array = df['t2m'].values
print(f"Successfully loaded {len(T_out_array)} hours of weather data!")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

# 0. Generate timestamps for the whole year
dates = pd.date_range(start='2019-01-01', periods=len(T_out_array), freq='h')

# 1. Declare building parameters
R_val = 0.005
C_val = 15000000
hp_power_val = 2000
target = 20.0
dt_val = 3600

# 2. Initiate ASHP instance
myASHP = ASHP(hp_power=hp_power_val, R=R_val, C=C_val, dt=dt_val)

# 3. RUN SIMULATION with +/- 1 degree deadband
T_in_dyn, P_in_dyn, COP_dyn = myASHP.run_ashp_simulation_advanced(
    T_out_data=T_out_array,
    T_min=target - 1.0,
    T_max=target + 1.0,
    flow_temp=45.0
)

# 4. PLOTTING WITH DATES
start_hour = 744
end_hour = 792
time_slice = dates[start_hour:end_hour]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1.plot(time_slice, T_in_dyn[start_hour:end_hour], label='Indoor Temp', color='green')
ax1.plot(time_slice, T_out_array[start_hour:end_hour], label='Outdoor Temp', color='blue', alpha=0.5)
ax1.axhspan(target - 1.0, target + 1.0, color='gray', alpha=0.2, label='Comfort Band')
ax1.axhline(y=target, color='gray', linestyle=':', label='Target (20C)')
ax1.set_ylabel('Temperature (C)')
ax1.legend()
ax1.grid(True)

ax2.step(time_slice, P_in_dyn[start_hour:end_hour] / 1000, label='Power (kW)', color='orange', where='post')
ax2.set_ylabel('Power (kW)')
ax2.set_xlabel('Date')

ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %d %H:%M'))
plt.xticks(rotation=45)

ax2.legend()
ax2.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.dates as mdates

# 0. Generate timestamps for the whole year (matching weather data length)
dates = pd.date_range(start='2019-01-01', periods=len(T_out_array), freq='h')

# =====================================================================
# SENSITIVITY ANALYSIS: Comparing 3 Levels of Insulation (R)
# =====================================================================

target = 20.0
t_min, t_max = target - 1.0, target + 1.0
base_C = 15000000
base_power = 2000
dt_val = 3600

# 1. Create three separate "Virtual Houses" with different R values
house_poor = ASHP(hp_power=base_power, R=0.002, C=base_C, dt=dt_val)
house_avg  = ASHP(hp_power=base_power, R=0.005, C=base_C, dt=dt_val)
house_good = ASHP(hp_power=base_power, R=0.010, C=base_C, dt=dt_val)

# 2. Run the Advanced simulation for all three houses
T_in_poor, P_in_poor, _ = house_poor.run_ashp_simulation_advanced(T_out_array, T_min=t_min, T_max=t_max)
T_in_avg,  P_in_avg,  _ = house_avg.run_ashp_simulation_advanced(T_out_array, T_min=t_min, T_max=t_max)
T_in_good, P_in_good, _ = house_good.run_ashp_simulation_advanced(T_out_array, T_min=t_min, T_max=t_max)

# ==========================================
# 3. PLOTTING THE COMPARISON (48-Hour Zoom)
# ==========================================
start_hour = 744
end_hour = 792
time_slice = dates[start_hour:end_hour]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# --- Top Graph: Indoor Temperatures ---
ax1.plot(time_slice, T_in_poor[start_hour:end_hour], label='Temp: Poor (R=0.002)', color='red', linewidth=2)
ax1.plot(time_slice, T_in_avg[start_hour:end_hour],  label='Temp: Average (R=0.005)', color='orange', linewidth=2)
ax1.plot(time_slice, T_in_good[start_hour:end_hour], label='Temp: Good (R=0.010)', color='green', linewidth=2)

ax1.plot(time_slice, T_out_array[start_hour:end_hour], label='Outdoor Temp', color='blue', alpha=0.3, linestyle='--')
ax1.axhspan(t_min, t_max, color='gray', alpha=0.1, label='Comfort Band')
ax1.axhline(y=target, color='gray', linestyle=':', label='Target (20C)')

ax1.set_ylabel('Indoor Temperature (°C)', fontsize=12)
ax1.set_title('Sensitivity Analysis: Impact of Thermal Resistance', fontsize=14)
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# --- Bottom Graph: Grid Power Demand ---
ax2.step(time_slice, P_in_poor[start_hour:end_hour] / 1000, label='Power: Poor', color='red', alpha=0.6, where='post')
ax2.step(time_slice, P_in_avg[start_hour:end_hour] / 1000,  label='Power: Average', color='orange', alpha=0.6, where='post')
ax2.step(time_slice, P_in_good[start_hour:end_hour] / 1000, label='Power: Good', color='green', alpha=0.8, where='post')

ax2.set_ylabel('Electrical Power (kW)', fontsize=12)
ax2.set_xlabel('Date', fontsize=12)
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

# Format the x-axis for dates
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %d %H:%M'))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# 4. PRINT SUMMARY
energy_poor = np.sum(P_in_poor) / 1000
energy_avg  = np.sum(P_in_avg) / 1000
energy_good = np.sum(P_in_good) / 1000

print(f"--- ANNUAL ENERGY CONSUMPTION (kWh) ---")
print(f"Poor Insulation House: {energy_poor:,.0f} kWh")
print(f"Average Insulation House: {energy_avg:,.0f} kWh")
print(f"Good Insulation House: {energy_good:,.0f} kWh")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ipywidgets as widgets
from ipywidgets import interact

# Generate timestamps matching your weather data
dates = pd.date_range(start='2019-01-01', periods=len(T_out_array), freq='h')

@interact(
    R_val=widgets.FloatSlider(value=0.005, min=0.001, max=0.015, step=0.001, description='Insulation (R):', readout_format='.3f'),
    C_val=widgets.IntSlider(value=15000000, min=5000000, max=30000000, step=1000000, description='Thermal Mass:'),
    hp_power=widgets.IntSlider(value=2000, min=1000, max=5000, step=250, description='HP Power (W):'),
    target=widgets.FloatSlider(value=20.0, min=15.0, max=25.0, step=0.5, description='Target Temp:')
)
def interactive_simulation(R_val, C_val, hp_power, target):
    dt_val = 3600
    interactive_house = ASHP(hp_power=hp_power, R=R_val, C=C_val, dt=dt_val)

    t_min = target - 1.0
    t_max = target + 1.0

    T_in, P_in, _ = interactive_house.run_ashp_simulation_advanced(
        T_out_data=T_out_array, T_min=t_min, T_max=t_max, flow_temp=45.0)

    start_hour = 744
    end_hour = 792
    time_slice = dates[start_hour:end_hour]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    ax1.plot(time_slice, T_in[start_hour:end_hour], label='Indoor Temp', color='green', linewidth=2)
    ax1.plot(time_slice, T_out_array[start_hour:end_hour], label='Outdoor Temp', color='blue', alpha=0.3, linestyle='--')
    ax1.axhspan(t_min, t_max, color='gray', alpha=0.2, label='Comfort Band')
    ax1.axhline(y=target, color='gray', linestyle=':', label='Target Setpoint')

    ax1.set_ylabel('Temperature (°C)')
    ax1.set_title(f'Interactive Advanced Simulation: R={R_val:.3f}, C={C_val:,}, Power={hp_power}W')
    ax1.legend(loc='lower right')
    ax1.grid(True, alpha=0.5)

    ax2.step(time_slice, P_in[start_hour:end_hour] / 1000, label='Grid Power', color='orange', where='post')
    ax2.set_ylabel('Power (kW)')
    ax2.set_ylim(0, 6)
    ax2.legend(loc='upper right')
    ax2.grid(True, alpha=0.5)

    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %d %H:%M'))
    for tick in ax2.get_xticklabels():
        tick.set_rotation(45)

    plt.tight_layout()
    plt.show()

    annual_energy = np.sum(P_in) / 1000
    print(f"Total Annual Grid Energy Required: {annual_energy:,.0f} kWh")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define our ranges
ambient_temps = np.arange(-10, 21, 2)  # -10°C to 20°C
flow_temps = np.arange(30, 61, 5)      # 30°C to 60°C (Underfloor to Radiators)

cop_matrix = np.zeros((len(flow_temps), len(ambient_temps)))

# Calculate theoretical COP with a 0.4 efficiency factor
for i, f_temp in enumerate(flow_temps):
    for j, a_temp in enumerate(ambient_temps):
        T_hot_K = f_temp + 273.15
        T_cold_K = a_temp + 273.15
        if T_hot_K > T_cold_K:
            cop_matrix[i, j] = min(5.0, max(1.0, 0.4 * (T_hot_K / (T_hot_K - T_cold_K))))
        else:
            cop_matrix[i, j] = 5.0

plt.figure(figsize=(10, 6))
sns.heatmap(cop_matrix, annot=True, fmt=".2f", cmap="YlOrRd_r",
            xticklabels=ambient_temps, yticklabels=flow_temps)
plt.title("Sensitivity Matrix: Impact of Ambient & Flow Temp on COP")
plt.xlabel("Outside Ambient Temperature (°C)")
plt.ylabel("System Flow Temperature (°C)")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define parameters for the diagram
T_min = 19.0
T_max = 21.0
P_max = 5000  # Max compressor power in Watts
k_p = 2500    # Proportional gain

# Generate temperature array (X-axis)
temperatures = np.linspace(15, 23, 500)
power_draw = []

# Calculate power draw based on your Section 3.4 logic
for T_in in temperatures:
    if T_in < T_min:
        # Proportional ramp up, capped at P_max
        p = min(k_p * (T_min - T_in), P_max)
        power_draw.append(p)
    elif T_min <= T_in <= T_max:
        # Inside deadband (simulating a low steady-state heat loss for the diagram)
        power_draw.append(800)
    else:
        # Above T_max, compressor is off
        power_draw.append(0)

# Create the plot
plt.figure(figsize=(10, 6))
plt.plot(temperatures, power_draw, linewidth=2.5, color='#1f77b4')

# Add boundary lines and shading for the comfort deadband
plt.axvline(x=T_min, color='orange', linestyle='--', label='T_min (Lower Boundary)')
plt.axvline(x=T_max, color='red', linestyle='--', label='T_max (Upper Boundary)')
plt.axvspan(T_min, T_max, color='green', alpha=0.1, label='Comfort Deadband')

# Add labels and formatting
plt.title('ASHP Proportional Inverter Control Logic', fontsize=14, fontweight='bold')
plt.xlabel('Indoor Temperature (°C)', fontsize=12)
plt.ylabel('Compressor Electrical Power (W)', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.7)
plt.legend(loc='upper right', fontsize=10)

# Save and show
plt.tight_layout()
plt.savefig('ashp_control_diagram.png', dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Create the figure
fig, ax = plt.subplots(figsize=(7, 5))
ax.axis('off')

# Draw the main UML box
rect = patches.Rectangle((0.05, 0.05), 0.9, 0.9, linewidth=2, edgecolor='#2c3e50', facecolor='#f8f9fa')
ax.add_patch(rect)

# Draw the separator lines
plt.plot([0.05, 0.95], [0.82, 0.82], color='#2c3e50', lw=2)
plt.plot([0.05, 0.95], [0.45, 0.45], color='#2c3e50', lw=2)

# Add the Class Name (Centered)
plt.text(0.5, 0.87, 'ASHP', fontsize=18, fontweight='bold', ha='center', color='#2c3e50')

# Add the Attributes (Top Section)
attributes = [
    "- P_max : float",
    "- R : float",
    "- C : float",
    "- dt : int",
    "- T_flow : float",
    "- T_min : float",
    "- T_max : float"
]
for i, attr in enumerate(attributes):
    plt.text(0.1, 0.76 - (i * 0.045), attr, fontsize=12, fontfamily='monospace', color='#34495e')

# Add the Methods (Bottom Section)
methods = [
    "+ __init__(P_max, R, C, dt)",
    "+ calculate_cop(T_out, T_flow) : float",
    "+ proportional_control_logic(T_in, T_min, T_max) : float",
    "+ run_ashp_simulation_advanced(weather_array)"
]
for i, method in enumerate(methods):
    plt.text(0.1, 0.37 - (i * 0.07), method, fontsize=11, fontfamily='monospace', color='#34495e')

# Save the image
plt.tight_layout()
plt.savefig('ashp_uml_diagram.png', dpi=300, bbox_inches='tight')
plt.show()